# Lab 8.3 &mdash; Shared State & Message Passing

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Declare the team's shared state as a typed StateGraph schema
- Use a reducer so each node's partial update MERGES, not overwrites
- Run a real graph and watch both findings accumulate

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Message passing & shared state](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-08-03")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agents coordinate by sharing state (deck slide 7). In LangGraph the **state is a typed schema** you declare
once &mdash; a `TypedDict` &mdash; and it flows through every node. The key framework idea is the
**reducer**: annotate a field with `Annotated[list, add]` and LangGraph **merges** each node's partial update
instead of overwriting it. So two nodes can each return `{"findings": [...]}` and **both survive** &mdash; no
bespoke state class, no manual dict-merging. Keep the shared context **small &amp; relevant**; the reducer
does the plumbing every LangGraph agent relies on.

In [ ]:
# state = a typed schema; a reducer (Annotated[list, add]) merges each node's update.
print("no hand-rolled class -- the framework's StateGraph + reducers ARE the shared state")

## Build it
Complete the **reducer** on `findings` and the `tech_node` update. Then a real `StateGraph` runs
`billing -> tech` and the reducer accumulates both findings &mdash; no API key needed.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

# LangGraph's state IS the shared blackboard. Declare it once as a typed schema;
# a REDUCER says how each node's partial update is merged into it.
class TeamState(TypedDict):
    message: str
    findings: Annotated[list, ___]   # TODO: the reducer that MERGES successive/parallel updates (hint: operator.add)
    log: Annotated[list, add]

def billing_node(state):
    """A node reads shared state and returns a PARTIAL update -- the reducer merges it in."""
    return {"findings": ["billing: duplicate charge on 4471"], "log": [("billing", "looked up 4471")]}

def tech_node(state):
    return ___   # TODO: return tech's finding + a log entry, the same shape as billing_node

def build_graph():
    g = StateGraph(TeamState)
    g.add_node("billing", billing_node)
    g.add_node("tech", tech_node)
    g.add_edge(START, "billing")
    g.add_edge("billing", "tech")
    g.add_edge("tech", END)
    return g.compile()

try:
    # A REAL StateGraph runs offline (no model call): the reducer accumulates both nodes' findings.
    app = build_graph()
    final = app.invoke({"message": "charged twice and the app crashes", "findings": [], "log": []})
    print("findings:", final["findings"])
    print("log     :", final["log"])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `findings` is `Annotated[list, add]` &mdash; the **reducer**. Each node returns only its *partial* update `{"findings": [...]}`, and LangGraph merges them, so **nobody overwrites anybody**.
- This is the framework doing what a hand-rolled shared-state class used to do &mdash; but it's the real mechanism *every* LangGraph agent uses.
- The `log` (also reduced) preserves order &mdash; the seed of observability (Lab 8.10). Change the reducer and you change how updates combine.

## Your turn (open task &mdash; no grader)
Add a `status` field with a **custom reducer** &mdash; a function `(old, new) -> merged` that keeps only the
latest value &mdash; and a node that sets it. **What good looks like:** each node still returns a small partial
update, and your reducer decides how the shared state combines them (append vs overwrite) &mdash; the raw material
a synthesiser (Lab 8.9) turns into one reply.

---
### Key takeaway
LangGraph's typed state + reducers ARE the shared blackboard -- each node returns a small partial update and the reducer merges it. That replaces any hand-rolled state class, and it's the real mechanism every graph uses.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>